## 1. Install all required packages

In [1]:
%pip install torch torchvision torchaudio datasets transformers sacrebleu rouge_score nltk sentencepiece evaluate matplotlib

Note: you may need to restart the kernel to use updated packages.


## 2. Import all required libraries

In [2]:
import os
import torch
import nltk
import time
import json
import gc
import matplotlib.pyplot as plt
%matplotlib inline
import sacrebleu
import evaluate
from datasets import load_dataset
from rouge_score import rouge_scorer
from transformers import MarianTokenizer, MarianMTModel, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from torch.utils.data import DataLoader
from nltk.translate.meteor_score import meteor_score
from tqdm.auto import tqdm
import numpy as np
from IPython.display import display, HTML
import pandas as pd

os.environ["WANDB_DISABLED"] = "true"

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("punkt_tab")

/Users/manas/development/bla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to /Users/manas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/manas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/manas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
# Create results directory
import os
if not os.path.exists("./results"):
    os.makedirs("./results")
    print("Created results directory")
else:
    print("Results directory already exists")

Results directory already exists


## Variables being set for resuming training from last saved checkpoint after changing the notebook

In [4]:
# Set defaults for checkpoint resumption
RESUME_TRAINING = True  # Set to True to resume from checkpoint
OLD_CHECKPOINT_PATH = "./fine_tuned_opus-mt-en-hi"  
NEW_CHECKPOINT_PATH = "./results/checkpoint_epoch2_step52001"
TRANSITIONING_FROM_OLD_NOTEBOOK = False  

# Set the starting epoch explicitly
start_epoch = 1  # 0-indexed, so 1 means Epoch 2

## 3. Device setup

In [5]:
# Verify MPS availability
print(f"MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")
print(f"MPS built: {torch.backends.mps.is_built() if hasattr(torch.backends, 'mps') else False}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Set device with proper error handling
try:
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using MPS (Metal Performance Shaders) device")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using CUDA device")
    else:
        device = torch.device("cpu")
        print("Using CPU device")
except Exception as e:
    print(f"Error setting device: {e}")
    device = torch.device("cpu")
    print("Falling back to CPU")

MPS available: True
MPS built: True
CUDA available: False
Using MPS (Metal Performance Shaders) device


## 4. Loading dataset

In [6]:
# Load full dataset
dataset = load_dataset("cfilt/iitb-english-hindi")
train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [7]:
# After loading the dataset
print(f"Original dataset size check:")
print(f"Training split size: {len(dataset['train'])} examples")
print(f"Validation split size: {len(dataset['validation'])} examples")
print(f"Test split size: {len(dataset['test'])} examples")

# Checking a few random examples to make sure they're unique
import random
random.seed(42)  # For reproducibility
sample_indices = random.sample(range(len(dataset['train'])), 3)
print("\nSample examples:")
for i, idx in enumerate(sample_indices):
    print(f"Example {i+1} (index {idx}):")
    print(f"English: {dataset['train'][idx]['translation']['en']}")
    print(f"Hindi: {dataset['train'][idx]['translation']['hi']}")
    print()

Original dataset size check:
Training split size: 1659083 examples
Validation split size: 520 examples
Test split size: 2507 examples

Sample examples:
Example 1 (index 1340975):
English: When the banyan 's leaves are still pink and tender, the delicate Map butterfly comes to feast on the sweet sticky smear of syrup on them.
Hindi: जब बरगद के नए पत्ते गुलाबी और नर्म होते हैं, तब नाजुक सी मैप तितली उसके मधुर-चिपचिपे रस को चूसने आती है। 

Example 2 (index 233478):
English: First page:
Hindi: à¤ªà¥à¤°à¤µà¤¿à¤·à¥à¤à¤¿ à¤à¥à¤¡à¤¼à¥à¤

Example 3 (index 52451):
English: Whether to show popup notifications when away or busy.
Hindi: क्या पॉप अप अधिसूचना दिखानी है जब दूर या व्यस्त है. 



## Subset selection

In [8]:
# Create a subset of the training data
subset_percentage = 100  # Change to 10% or 20% as needed

# Calculate subset size and select random samples with fixed seed
subset_size = int(len(train_data) * subset_percentage / 100)
train_subset = train_data.shuffle(seed=42).select(range(subset_size))

# Print clear information about dataset sizes
print(f"Original training data size: {len(train_data):,} examples")
print(f"Using {subset_percentage}% subset: {len(train_subset):,} examples")
print(f"Validation data size: {len(val_data):,} examples")
print(f"Test data size: {len(test_data):,} examples")

# Replace full dataset with subset
train_data = train_subset

# Log the first few examples to verify
print("\nExample data from training subset:")
for i in range(min(3, len(train_subset))):
    print(f"Example {i+1}:")
    print(f"English: {train_subset[i]['translation']['en']}")
    print(f"Hindi: {train_subset[i]['translation']['hi']}")
    print("-" * 50)

Original training data size: 1,659,083 examples
Using 100% subset: 1,659,083 examples
Validation data size: 520 examples
Test data size: 2,507 examples

Example data from training subset:
Example 1:
English: on the intuition.
Hindi: अंतर्ज्ञान पर। 
--------------------------------------------------
Example 2:
English: Ceiba pentandra
Hindi: कण्टकारी
--------------------------------------------------
Example 3:
English: Have you not regarded how your Lord dealt with [the people of] ‘Ad,
Hindi: क्या तुमने देखा नहीं कि तुम्हारे आद के साथ क्या किया
--------------------------------------------------


## 5. Model and tokenizer initialization

In [9]:
# Initialize tokenizer and model with support for old checkpoint format
model_name = "Helsinki-NLP/opus-mt-en-hi"

# Check if we're transitioning from the old notebook
if RESUME_TRAINING and TRANSITIONING_FROM_OLD_NOTEBOOK and os.path.exists(OLD_CHECKPOINT_PATH):
    print(f"Loading model and tokenizer from old notebook checkpoint: {OLD_CHECKPOINT_PATH}")
    try:
        # Old notebook likely saved the model and tokenizer directly
        tokenizer = MarianTokenizer.from_pretrained(OLD_CHECKPOINT_PATH)
        model = MarianMTModel.from_pretrained(OLD_CHECKPOINT_PATH)
        print("Successfully loaded model and tokenizer from old notebook checkpoint")
    except Exception as e:
        print(f"Error loading from old checkpoint: {e}")
        print("Falling back to original model")
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name)
# Check if continuing with new format
elif RESUME_TRAINING and os.path.exists(f"{NEW_CHECKPOINT_PATH}/model"):
    print(f"Loading model and tokenizer from new checkpoint: {NEW_CHECKPOINT_PATH}")
    try:
        tokenizer = MarianTokenizer.from_pretrained(f"{NEW_CHECKPOINT_PATH}/tokenizer")
        model = MarianMTModel.from_pretrained(f"{NEW_CHECKPOINT_PATH}/model")
        print("Successfully loaded model and tokenizer from new checkpoint")
    except Exception as e:
        print(f"Error loading from checkpoint: {e}")
        print("Falling back to original model")
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name)
else:
    print("Loading original pre-trained model")
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)

# Move model to device
model.to(device)

Loading model and tokenizer from new checkpoint: ./results/checkpoint_epoch2_step52001
Successfully loaded model and tokenizer from new checkpoint


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(61950, 512, padding_idx=61949)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(61950, 512, padding_idx=61949)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

## 6. Pre-processing function

In [10]:
# Preprocessing function
def preprocess_function(batch):
    source_texts = [ex["en"] for ex in batch["translation"]]
    target_texts = [ex["hi"] for ex in batch["translation"]]
    model_inputs = tokenizer(source_texts, truncation=True, padding="max_length", max_length=128)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(target_texts, truncation=True, padding="max_length", max_length=128)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

## 7. Data Tokenization

In [11]:
# Additional optimization for faster tensor creation
from functools import partial
import numpy as np  # Make sure this is imported

def optimized_map_function(batch, tokenizer):
    source_texts = [ex["en"] for ex in batch["translation"]]
    target_texts = [ex["hi"] for ex in batch["translation"]]
    
    # Tokenize inputs
    model_inputs = tokenizer(source_texts, truncation=True, padding="max_length", max_length=128)
    
    # Tokenize targets using target tokenizer
    with tokenizer.as_target_tokenizer():
        targets = tokenizer(target_texts, truncation=True, padding="max_length", max_length=128)
    
    # Convert to numpy arrays first before creating tensors later
    model_inputs["labels"] = np.array(targets["input_ids"])
    
    return model_inputs

# ONLY use the optimized tokenization code
print("Starting optimized dataset tokenization...")
start_time = time.time()

# Create a partial function with the tokenizer
map_fn = partial(optimized_map_function, tokenizer=tokenizer)

# Apply the function - creates numpy arrays instead of lists
tokenized_train = train_data.map(map_fn, batched=True, batch_size=1000)
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

tokenized_val = val_data.map(map_fn, batched=True, batch_size=1000)
tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Optimized dataset tokenization completed in {time.time() - start_time:.2f} seconds")
print(f"Tokenized train dataset size: {len(tokenized_train)}")
print(f"Tokenized validation dataset size: {len(tokenized_val)}")

Starting optimized dataset tokenization...


Map:   0%|          | 0/1659083 [00:00<?, ? examples/s]/Users/manas/development/bla/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 1659083/1659083 [03:53<00:00, 7100.81 examples/s]

Optimized dataset tokenization completed in 234.04 seconds
Tokenized train dataset size: 1659083
Tokenized validation dataset size: 520


## 8. Memory Optimization

In [12]:
# Memory optimization for large datasets
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [13]:
def optimize_mps_memory():
    """Optimize memory usage on MPS devices"""
    if torch.backends.mps.is_available():
        # Force garbage collection
        import gc
        gc.collect()
        
        # Release unused memory if possible
        if hasattr(torch.mps, 'empty_cache'):
            torch.mps.empty_cache()
        
        # Ensure deterministic behavior is off (can cause MPS issues)
        torch.use_deterministic_algorithms(False)
        
        print("MPS memory optimized")

# Calling this function periodically during training
# Adding for training loop after each epoch

## 9. Setting up data collator

In [14]:
# Optimized data collator to avoid slow tensor creation
class OptimizedDataCollatorForSeq2Seq(DataCollatorForSeq2Seq):
    def __call__(self, features, return_tensors=None):
        # Process input features as usual
        batch = super().__call__(features, return_tensors=None)
        
        # Specifically handle labels which is causing the warning
        if "labels" in batch and isinstance(batch["labels"], list):
            try:
                # Convert list to numpy array first, then to tensor
                import numpy as np
                labels_array = np.array(batch["labels"])
                batch["labels"] = torch.tensor(labels_array, dtype=torch.int64)
            except Exception as e:
                print(f"Optimization failed: {e}, falling back to default")
        
        # Handle other fields
        for key, value in batch.items():
            if key != "labels" and isinstance(value, list):
                try:
                    import numpy as np
                    array_value = np.array(value)
                    batch[key] = torch.tensor(array_value)
                except:
                    pass  # Fall back to default behavior
        
        # Apply padding if needed
        if return_tensors is not None:
            batch = {k: torch.tensor(v) if not isinstance(v, torch.Tensor) else v 
                     for k, v in batch.items()}
        
        return batch

# Replace your data collator with the optimized version
data_collator = OptimizedDataCollatorForSeq2Seq(
    tokenizer, 
    model=model,
    padding=True,
    return_tensors="pt"
)

# Print tokenized dataset info
print(f"Tokenized train dataset size: {len(tokenized_train)}")
print(f"Tokenized validation dataset size: {len(tokenized_val)}")

Tokenized train dataset size: 1659083
Tokenized validation dataset size: 520


## 10. Setting up training arguments

In [15]:
# Training arguments for full dataset fine-tuning
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    predict_with_generate=True,
    # Use fp16 only on CUDA devices
    fp16=torch.cuda.is_available(),
    # Set reasonable defaults for dataloader
    dataloader_num_workers=4 if torch.cuda.is_available() else 0,
    dataloader_pin_memory=False,
    # Logging
    logging_dir="./logs",
    logging_strategy="epoch",
    report_to=["tensorboard"]
)

In [16]:
# # After you've created your train_dataloader
# print(f"Number of training batches: {len(train_dataloader)}")
# print(f"Batch size: {training_args.per_device_train_batch_size}")
# print(f"Total training examples: {len(train_dataloader) * training_args.per_device_train_batch_size}")

# # And for validation
# if 'eval_dataloader' in locals() or 'eval_dataloader' in globals():
#     print(f"Number of validation batches: {len(eval_dataloader)}")
#     print(f"Total validation examples: {len(eval_dataloader) * training_args.per_device_eval_batch_size}")
# else:
#     print(f"Number of validation examples: {len(tokenized_val)}")

## 11. Checkpoint Management utility functions

In [17]:
# Checkpoint management utility functions
def save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, metrics, path):
    """Save model checkpoint with all training state"""
    if not os.path.exists(path):
        os.makedirs(path)
    
    # Save model and tokenizer
    model.save_pretrained(f"{path}/model")
    tokenizer.save_pretrained(f"{path}/tokenizer")
    
    # Save optimizer and scheduler states
    checkpoint = {
        'epoch': epoch,
        'step': step,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'metrics': metrics,
        'args': training_args.to_dict()
    }
    torch.save(checkpoint, f"{path}/training_state.pt")
    print(f"Checkpoint saved to {path}")

def load_checkpoint(model, tokenizer, optimizer, scheduler, path):
    """Load model checkpoint with all training state"""
    # Load model and tokenizer
    model = model.from_pretrained(f"{path}/model")
    tokenizer = tokenizer.from_pretrained(f"{path}/tokenizer")
    
    # Load optimizer and scheduler states if available
    if os.path.exists(f"{path}/training_state.pt"):
        checkpoint = torch.load(f"{path}/training_state.pt")
        optimizer.load_state_dict(checkpoint['optimizer'])
        if scheduler and checkpoint['scheduler']:
            scheduler.load_state_dict(checkpoint['scheduler'])
        start_epoch = checkpoint['epoch']
        global_step = checkpoint['step']
        print(f"Loaded checkpoint from {path} (epoch {start_epoch}, step {global_step})")
        return model, tokenizer, optimizer, scheduler, start_epoch, global_step
    
    return model, tokenizer, optimizer, scheduler, 0, 0

## Signal handler for saving on interruption

In [22]:
# Signal handler for manual interruption
import signal
import sys
import inspect

def signal_handler(sig, frame):
    print("\n\nTraining interrupted. Saving checkpoint...")
    try:
        # Create an emergency save path
        emergency_path = "./results/emergency_checkpoint"
        
        # Check for globally defined model and tokenizer
        if 'model' not in globals() or 'tokenizer' not in globals():
            print("Model or tokenizer not available, cannot save checkpoint.")
            sys.exit(0)
            
        # Try to find the training loop frame to access variables
        current_model = globals().get('model')
        current_tokenizer = globals().get('tokenizer')
        
        # Look for variables in the custom_training_loop scope
        current_optimizer = None
        current_epoch = 0
        current_global_step = 0
        current_metrics = {}
        
        # First check globals for signal handler variables
        if 'current_optimizer' in globals():
            current_optimizer = globals().get('current_optimizer')
        if 'current_epoch' in globals():
            current_epoch = globals().get('current_epoch')
        if 'current_global_step' in globals():
            current_global_step = globals().get('current_global_step')
        if 'current_metrics' in globals():
            current_metrics = globals().get('current_metrics')
        
        # Create directory
        import os
        os.makedirs(emergency_path, exist_ok=True)
        
        # Call the save_checkpoint function
        if 'save_checkpoint' in globals():
            save_checkpoint(
                current_model, 
                current_tokenizer, 
                current_optimizer, 
                globals().get('current_scheduler'),
                current_epoch, 
                current_global_step, 
                current_metrics,
                emergency_path
            )
            print(f"Emergency checkpoint saved to {emergency_path}")
        else:
            # Manual save if save_checkpoint not available
            current_model.save_pretrained(f"{emergency_path}/model")
            current_tokenizer.save_pretrained(f"{emergency_path}/tokenizer")
            print("Model and tokenizer saved (no training state)")
    except Exception as e:
        print(f"Error saving emergency checkpoint: {e}")
        # Try a simpler save as fallback
        try:
            if 'model' in globals():
                globals()['model'].save_pretrained(f"{emergency_path}/model_fallback")
                globals()['tokenizer'].save_pretrained(f"{emergency_path}/tokenizer_fallback")
                print("Fallback save completed")
        except Exception as e2:
            print(f"Fallback save also failed: {e2}")
    
    sys.exit(0)

# Register the signal handler
signal.signal(signal.SIGINT, signal_handler)

<function _signal.default_int_handler(signalnum, frame, /)>

## 12. Defining Evaluation Metrics

In [23]:
# Unified metrics computation function
def compute_metrics(predictions, references, tokenized_references=False):
    """
    Compute BLEU, ROUGE, and METEOR scores
    
    Args:
        predictions: List of prediction strings
        references: List of reference strings or list of lists for BLEU
        tokenized_references: Whether references are already tokenized for BLEU
    """
    # Flatten predictions and references if needed
    if isinstance(references[0], list) and len(references[0]) == 1:
        references = [ref[0] for ref in references]
    
    # Set up metrics
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")

    # Ensure references are in correct format for different metrics
    bleu_refs = references if tokenized_references else [[ref] for ref in references]
    rouge_refs = [ref[0] if isinstance(ref, list) else ref for ref in references]
    
    # Compute BLEU score
    bleu_score = bleu_metric.compute(predictions=predictions, references=bleu_refs)["bleu"]
    
    # Compute ROUGE scores
    rouge_scores = rouge_metric.compute(predictions=predictions, references=rouge_refs)
    rouge_1 = rouge_scores["rouge1"]
    rouge_2 = rouge_scores["rouge2"]
    rouge_l = rouge_scores["rougeL"]
    
    # Compute METEOR score
    try:
        meteor_scores = [meteor_score([ref.split()], pred.split()) 
                         for pred, ref in zip(predictions, rouge_refs)]
        meteor_avg = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0
    except Exception as e:
        print(f"Warning: METEOR calculation failed - {e}")
        meteor_avg = 0
    
    results = {
        "BLEU": round(bleu_score * 100, 2),
        "ROUGE-1": round(rouge_1 * 100, 2),
        "ROUGE-2": round(rouge_2 * 100, 2),
        "ROUGE-L": round(rouge_l * 100, 2),
        "METEOR": round(meteor_avg * 100, 2)
    }
    
    return results

## 13. Model training

In [24]:
# Custom training loop with improved MPS optimization
from torch.optim import AdamW
from transformers import get_scheduler
import torch.nn.functional as F

def custom_training_loop(start_epoch=0):
    """
    Custom training loop optimized for MPS with proper checkpointing
    """
    # Expose key variables as globals for the signal handler
    global current_epoch, current_global_step, current_optimizer, current_scheduler, current_metrics
    
    # Initialize global variables for signal handler
    current_epoch = start_epoch
    current_global_step = 0
    current_metrics = {
        "train_loss": [],
        "eval_loss": [],
        "bleu": [],
        "rouge1": [],
        "rouge2": [],
        "rougeL": [],
        "meteor": []
    }
    
    print(f"Starting training on {device} device...")
    
    # MPS-specific optimizations
    mps_device = device.type == 'mps'
    if mps_device:
        print("Using MPS-optimized training loop")
        # Important: These settings improve MPS performance
        torch.backends.cudnn.benchmark = False  # Important for MPS
        torch.use_deterministic_algorithms(False)  # Some ops aren't deterministic on MPS
    
    # Setup optimizer and learning rate scheduler
    optimizer = AdamW(model.parameters(), lr=training_args.learning_rate, weight_decay=training_args.weight_decay)
    current_optimizer = optimizer  # Update global for signal handler
    
    # Create train dataloader with proper parameters for MPS
    train_dataloader = DataLoader(
        tokenized_train, 
        batch_size=training_args.per_device_train_batch_size, 
        shuffle=True, 
        collate_fn=data_collator,
        num_workers=0,  # Must be 0 for MPS
        pin_memory=False  # Must be False for MPS
    )
    
    # Calculate total steps for lr scheduler
    num_epochs = int(training_args.num_train_epochs)
    total_batches = len(train_dataloader)
    total_steps = total_batches * num_epochs // training_args.gradient_accumulation_steps
    
    # Create scheduler
    scheduler = get_scheduler(
        "linear", 
        optimizer=optimizer, 
        num_warmup_steps=0,
        num_training_steps=total_steps
    )
    current_scheduler = scheduler  # Update global for signal handler
    
    # Initialize tracking variables
    global_step = 0
    checkpoint_path = "./results/latest_checkpoint"
    all_metrics = current_metrics.copy()  # Initialize metrics dictionary
    
    # Load checkpoint if available - More robust implementation
    if RESUME_TRAINING:
        try:
            # Handle transitioning from old notebook format
            if TRANSITIONING_FROM_OLD_NOTEBOOK and os.path.exists(OLD_CHECKPOINT_PATH):
                print("Starting from epoch 1 since transitioning from old notebook checkpoint")
                # start_epoch already set via function parameter
                # Old notebook doesn't have training state, so we use the loaded model but fresh optimizer
            
            # Handle new checkpoint format
            elif os.path.exists(NEW_CHECKPOINT_PATH):
                print(f"Attempting to load checkpoint from {NEW_CHECKPOINT_PATH}")
                
                # Try to load training state
                if os.path.exists(f"{NEW_CHECKPOINT_PATH}/training_state.pt"):
                    try:
                        checkpoint = torch.load(f"{NEW_CHECKPOINT_PATH}/training_state.pt")
                        if checkpoint and isinstance(checkpoint, dict):
                            if 'optimizer' in checkpoint and checkpoint['optimizer']:
                                optimizer.load_state_dict(checkpoint['optimizer'])
                                print("Optimizer state loaded")
                            if 'scheduler' in checkpoint and checkpoint['scheduler']:
                                scheduler.load_state_dict(checkpoint['scheduler'])
                                print("Scheduler state loaded")
                            if 'epoch' in checkpoint:
                                start_epoch = checkpoint['epoch']
                                current_epoch = start_epoch  # Update global for signal handler
                                print(f"Resuming from epoch {start_epoch}")
                            if 'step' in checkpoint:
                                global_step = checkpoint['step']
                                current_global_step = global_step  # Update global for signal handler
                                print(f"Resuming from step {global_step}")
                            if 'metrics' in checkpoint:
                                all_metrics = checkpoint['metrics']
                                current_metrics = all_metrics.copy()  # Update global for signal handler
                                print("Metrics history loaded")
                    except Exception as e:
                        print(f"Failed to load training state: {e}")
        except Exception as e:
            print(f"Error during checkpoint loading: {e}")
    
    # Create validation dataloader - FIXED
    eval_dataloader = DataLoader(
        tokenized_val, 
        batch_size=training_args.per_device_eval_batch_size, 
        collate_fn=data_collator,  # This is crucial - uses the same collator
        num_workers=0,  # Must be 0 for MPS
        pin_memory=False,  # Must be False for MPS
        shuffle=False
    )
    
    # MPS memory management function - improved
    if not hasattr(torch.mps, 'empty_cache'):
        def empty_cache(): 
            pass
        torch.mps.empty_cache = empty_cache
        
    def mps_empty_cache():
        """Optimized memory cleanup for MPS devices"""
        if device.type == 'mps':
            # For MPS (Apple Silicon), this helps free memory
            gc.collect()
            if hasattr(torch.mps, 'empty_cache'):
                torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    best_bleu = 0
    checkpoint_counter = 0
    update_freq = 50  # Update progress bar every 50 batches
    
    # Training loop
    for epoch in range(start_epoch, num_epochs):
        # Update global for signal handler at start of epoch
        current_epoch = epoch
        
        model.train()
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # Track loss for this epoch
        train_losses = []
        
        # Use tqdm for progress tracking with optimized updates
        progress_bar = tqdm(
            total=total_batches, 
            desc=f"Epoch {epoch+1}/{num_epochs}",
            dynamic_ncols=True,
            position=0,
            leave=True
        )
        
        # Reset gradients at start of epoch
        optimizer.zero_grad()
        accumulated_steps = 0
        
        # Main training loop
        for step, batch in enumerate(train_dataloader):
            # Calculate global step
            current_step = global_step + step + 1
            current_global_step = current_step  # Update global for signal handler
            checkpoint_counter += 1
            
            try:
                # Process batch - move to device
                batch = {k: v.to(device) for k, v in batch.items()}
                
                # Forward pass with MPS error handling
                try:
                    outputs = model(**batch)
                    loss = outputs.loss / training_args.gradient_accumulation_steps
                except RuntimeError as e:
                    if "MPS" in str(e) or "not implemented for" in str(e):
                        print(f"\nMPS error encountered: {e}. Moving batch to CPU.")
                        # Try to recover by moving to CPU
                        cpu_batch = {k: v.cpu() for k, v in batch.items()}
                        model_cpu = model.cpu()
                        outputs = model_cpu(**cpu_batch)
                        loss = outputs.loss / training_args.gradient_accumulation_steps
                        # Move model back to MPS
                        model.to(device)
                    else:
                        raise e
                
                # Backward pass
                loss.backward()
                
                # Track loss (but not too frequently to save computation)
                if step % 50 == 0:  # Only track loss every 50 steps
                    train_losses.append(loss.item() * training_args.gradient_accumulation_steps)
                
                if step % 5000 == 0:  # Every 5000 steps
                    # Save a snapshot of current metrics
                    interim_metrics_path = f"./results/interim_metrics_epoch{epoch+1}_step{current_step}"
                    os.makedirs(interim_metrics_path, exist_ok=True)
                    torch.save({'metrics': all_metrics}, f"{interim_metrics_path}/interim_metrics.pt")
                
                # Update progress bar less frequently for better performance
                if step % update_freq == 0 or step == total_batches - 1:
                    avg_loss = sum(train_losses[-100:]) / min(len(train_losses), 100) if train_losses else 0
                    # Calculate number of steps to update
                    update_steps = min(update_freq, step - progress_bar.n + 1) if step > 0 else 0
                    progress_bar.update(update_steps)
                    progress_bar.set_postfix({
                        "loss": f"{avg_loss:.4f}",
                        "batch": f"{step}/{total_batches}"
                    })
                
                # Update weights when we reach the right number of accumulated gradients
                accumulated_steps += 1
                if accumulated_steps == training_args.gradient_accumulation_steps:
                    # Clip gradients
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    
                    # Update parameters
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    accumulated_steps = 0
                    
                    # MPS-specific memory cleanup every 100 updates
                    if step % 100 == 0:
                        mps_empty_cache()
                    
                    # Save checkpoint every 5000 steps
                    if checkpoint_counter >= 5000:
                        save_path = f"./results/checkpoint_epoch{epoch+1}_step{current_step}"
                        try:
                            save_checkpoint(
                                model, tokenizer, optimizer, scheduler,
                                epoch, current_step, all_metrics, save_path
                            )
                            print(f"\nIntermediate checkpoint saved at step {current_step}")
                        except Exception as e:
                            print(f"Failed to save checkpoint: {e}")
                        checkpoint_counter = 0
            
            except Exception as e:
                print(f"\nError in training step {step}: {e}")
                continue
        
        # Update progress bar to completion
        progress_bar.update(total_batches - progress_bar.n)
        progress_bar.close()
        
        # Calculate average loss for the epoch
        if train_losses:
            avg_epoch_loss = sum(train_losses) / len(train_losses)
            all_metrics["train_loss"].append(avg_epoch_loss)
            current_metrics["train_loss"] = all_metrics["train_loss"].copy()  # Update global for signal handler
            print(f"Epoch {epoch+1}/{num_epochs} - Average training loss: {avg_epoch_loss:.4f}")
        
        # Memory cleanup
        mps_empty_cache()
        
        # Evaluation after each epoch
        print("Evaluating...")
        model.eval()

        eval_losses = []
        all_preds = []
        all_refs = []

        with torch.no_grad():
            eval_progress = tqdm(
                total=len(eval_dataloader),
                desc="Evaluation",
                position=0,
                leave=True
            )
            
            for batch_idx, eval_batch in enumerate(eval_dataloader):
                try:
                    # Move batch to device
                    eval_inputs = {k: v.to(device) for k, v in eval_batch.items()}
                    
                    # Forward pass with MPS error handling
                    try:
                        outputs = model(**eval_inputs)
                        eval_losses.append(outputs.loss.item())
                    
                        # Generate translations
                        generated_tokens = model.generate(
                            eval_inputs["input_ids"],
                            attention_mask=eval_inputs["attention_mask"],
                            max_length=128
                        )
                    except RuntimeError as e:
                        if "MPS" in str(e) or "not implemented for" in str(e):
                            print(f"\nMPS error during evaluation: {e}. Moving to CPU.")
                            # Try on CPU
                            cpu_batch = {k: v.cpu() for k, v in eval_batch.items()}
                            model_cpu = model.cpu()
                            outputs = model_cpu(**cpu_batch)
                            eval_losses.append(outputs.loss.item())
                            
                            generated_tokens = model_cpu.generate(
                                cpu_batch["input_ids"],
                                attention_mask=cpu_batch["attention_mask"],
                                max_length=128
                            )
                            # Move model back to MPS
                            model.to(device)
                        else:
                            raise e
                    
                    # Decode predictions and references
                    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
                    decoded_refs = tokenizer.batch_decode(eval_inputs["labels"], skip_special_tokens=True)
                    
                    # Add to collections
                    all_preds.extend(decoded_preds)
                    all_refs.extend(decoded_refs)
                    
                    # Update progress bar
                    eval_progress.update(1)
                        
                except Exception as e:
                    print(f"\nError in evaluation batch: {e}")
                    continue
            
            # Ensure progress bar completes
            eval_progress.close()
        
        # Calculate evaluation metrics
        try:
            eval_metrics = compute_metrics(all_preds, all_refs)
            eval_metrics["eval_loss"] = sum(eval_losses) / len(eval_losses) if eval_losses else float('inf')
            
            # Update tracking metrics
            all_metrics["eval_loss"].append(eval_metrics["eval_loss"])
            all_metrics["bleu"].append(eval_metrics["BLEU"])
            all_metrics["rouge1"].append(eval_metrics["ROUGE-1"])
            all_metrics["rouge2"].append(eval_metrics["ROUGE-2"])
            all_metrics["rougeL"].append(eval_metrics["ROUGE-L"])
            all_metrics["meteor"].append(eval_metrics["METEOR"])
            
            # Update global metrics for signal handler
            current_metrics = all_metrics.copy()
            
            # Print summary
            print(f"Epoch {epoch+1} - Eval Loss: {eval_metrics['eval_loss']:.4f}")
            print(f"BLEU: {eval_metrics['BLEU']:.4f}, ROUGE-L: {eval_metrics['ROUGE-L']:.4f}")
        
            # Save checkpoint after each epoch
            epoch_path = f"./results/checkpoint_epoch{epoch+1}_final"
            save_checkpoint(
                model, tokenizer, optimizer, scheduler,
                epoch+1, global_step+len(train_dataloader), all_metrics, epoch_path
            )
            
            # Save latest checkpoint (for resuming)
            save_checkpoint(
                model, tokenizer, optimizer, scheduler,
                epoch+1, global_step+len(train_dataloader), all_metrics, checkpoint_path
            )
            
            # Save best model
            if eval_metrics["BLEU"] > best_bleu:
                best_bleu = eval_metrics["BLEU"]
                best_path = f"./results/best_model"
                save_checkpoint(
                    model, tokenizer, optimizer, scheduler,
                    epoch+1, global_step+len(train_dataloader), all_metrics, best_path
                )
                print(f"New best model saved with BLEU: {best_bleu:.4f}")
        
        except Exception as e:
            print(f"Error during evaluation metrics calculation: {e}")
        
        # Update global step
        global_step += len(train_dataloader)
        current_global_step = global_step  # Update global for signal handler

        # For memory optimization after every epoch
        optimize_mps_memory()
    
    # Save final model
    try:
        final_path = "./results/final_model"
        save_checkpoint(
            model, tokenizer, optimizer, scheduler,
            num_epochs, global_step, all_metrics, final_path
        )
        print(f"Training complete! Final model saved to {final_path}")
    except Exception as e:
        print(f"Failed to save final model: {e}")
    
    return model, all_metrics

In [25]:
# Execute training
model, training_metrics = custom_training_loop(start_epoch)

# Save the fine-tuned model and tokenizer
model.save_pretrained("fine_tuned_opus-mt-en-hi")
tokenizer.save_pretrained("fine_tuned_opus-mt-en-hi")

Starting training on mps device...
Using MPS-optimized training loop
Attempting to load checkpoint from ./results/checkpoint_epoch2_step52001
Optimizer state loaded
Scheduler state loaded
Resuming from epoch 1
Resuming from step 52001
Metrics history loaded

Epoch 2/5


Epoch 2/5:   2%|▏         | 4950/207386 [20:54<12:22:50,  4.54it/s, loss=0.3230, batch=4950/207386]

Checkpoint saved to ./results/checkpoint_epoch2_step57001

Intermediate checkpoint saved at step 57001


Epoch 2/5:   5%|▍         | 9950/207386 [37:40<10:40:48,  5.14it/s, loss=0.3275, batch=9950/207386]

Checkpoint saved to ./results/checkpoint_epoch2_step62001

Intermediate checkpoint saved at step 62001


...

Checkpoint saved to ./results/checkpoint_epoch3_step302003

Intermediate checkpoint saved at step 302003


Epoch 3/5:  23%|██▎       | 47600/207386 [2:45:13<8:35:11,  5.17it/s, loss=0.2843, batch=47600/207386]

Checkpoint saved to ./results/checkpoint_epoch3_step307003

Intermediate checkpoint saved at step 307003


Epoch 3/5:  25%|██▌       | 51950/207386 [2:59:23<8:25:26,  5.13it/s, loss=0.2841, batch=51950/207386]

KeyboardInterrupt: 

## 14. Reloading fine tuned model

In [26]:
# Reload fine-tuned model - try multiple locations in order of preference
model_paths = [
    "fine_tuned_opus-mt-en-hi",  # Final saved model
    "./results/final_model/model",  # Final model checkpoint
    "./results/best_model/model",   # Best model based on metrics
    "./results/latest_checkpoint/model",  # Latest checkpoint
    NEW_CHECKPOINT_PATH + "/model"  # Specific checkpoint defined at top
]

model = None
tokenizer = None

# Try each path in order
for path in model_paths:
    try:
        print(f"Attempting to load model from {path}")
        model = MarianMTModel.from_pretrained(path)
        tokenizer_path = path.replace("/model", "/tokenizer") if "/model" in path else path
        tokenizer = MarianTokenizer.from_pretrained(tokenizer_path)
        
        # Handle device placement safely
        try:
            model.to(device)
        except RuntimeError as e:
            if "only available on CPU" in str(e):
                print("Warning: MPS-specific issue detected, falling back to CPU for model loading")
                device = torch.device('cpu')
                model.to(device)
            else:
                raise e
                
        print(f"Model loaded successfully from {path} to {device}")
        break  # Exit the loop if successful
    except Exception as e:
        print(f"Failed to load model from {path}: {e}")

if model is None:
    print("Could not load fine-tuned model from any location. Using original pre-trained model.")
    model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-hi")
    tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")
    model.to(device)

Attempting to load model from fine_tuned_opus-mt-en-hi
Failed to load model from fine_tuned_opus-mt-en-hi: fine_tuned_opus-mt-en-hi is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
Attempting to load model from ./results/final_model/model
Failed to load model from ./results/final_model/model: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './results/final_model/model'. Use `repo_type` argument if needed.
Attempting to load model from ./results/best_model/model
Model loaded successfully from ./results/best_model/model to mps


## 15. Defining evaluation functions

In [27]:
# Model evaluation 
def evaluate_model(test_data, model, tokenizer, device, num_samples=None):
    """Evaluate model on test data"""
    model.eval()
    predictions, references = [], []
    
    # Use full test dataset if num_samples is None
    num_samples = len(test_data) if num_samples is None else min(num_samples, len(test_data))
    
    print(f"Evaluating model on {num_samples} test examples...")
    progress_bar = tqdm(range(num_samples))
    
    for i in range(num_samples):
        try:
            source_text = test_data[i]["translation"]["en"]
            reference_text = test_data[i]["translation"]["hi"]
            
            # Directly tokenize and move to device without using data_collator
            inputs = tokenizer(source_text, return_tensors="pt", truncation=True, padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Generate translation
            with torch.no_grad():
                outputs = model.generate(**inputs, max_length=128)
            
            # Decode output
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Store results
            predictions.append(generated_text)
            references.append([reference_text])
            
            # Update progress bar
            progress_bar.update(1)
            
        except Exception as e:
            print(f"Error processing example {i}: {e}")
    
    # Compute metrics
    if predictions and references:
        metrics = compute_metrics(predictions, references, tokenized_references=True)
    else:
        metrics = {"BLEU": 0, "ROUGE-1": 0, "ROUGE-2": 0, "ROUGE-L": 0, "METEOR": 0}
        print("No successful translations to evaluate")
    
    # Return both metrics and raw predictions/references for further analysis
    return metrics, predictions, references

## 16. Executing evaluation of the model

In [28]:
# Evaluate on test dataset
print("Evaluating model on test data...")
try:
    metrics, preds, refs = evaluate_model(test_data, model, tokenizer, device, num_samples=100)
    print("Evaluation Metrics (Fine-Tuned Model):", metrics)
except Exception as e:
    print(f"Error during evaluation: {e}")
    preds = []
    refs = []

# Show some examples
print("\nExample translations:")
if preds and len(preds) > 0:
    for i in range(min(5, len(preds))):
        print(f"Source: {test_data[i]['translation']['en']}")
        print(f"Reference: {refs[i][0]}")
        print(f"Prediction: {preds[i]}")
        print("-" * 50)
else:
    print("No successful translations to display.")

Evaluating model on test data...
Evaluating model on 100 test examples...


[nltk_data] Downloading package wordnet to /Users/manas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/manas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/manas/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
100%|██████████| 100/100 [01:15<00:00,  1.33it/s]

Evaluation Metrics (Fine-Tuned Model): {'BLEU': 15.93, 'ROUGE-1': np.float64(11.0), 'ROUGE-2': np.float64(3.0), 'ROUGE-L': np.float64(10.0), 'METEOR': 40.39}

Example translations:
Source: A black box in your car?
Reference: आपकी कार में ब्लैक बॉक्स?
Prediction: आपकी कार में एक ब्लैक बॉक्स?
--------------------------------------------------
Source: As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
Reference: जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।
Prediction: जैसा कि अमेरिका के सड़क योजनाकारों के लिए नकदी की खोज करने के लिए प्रयास करते हैं, बहुत से लोग एक छोटे से काले बक्से में एक समाधान देखने लगे हैं जो आपकी कार के डेढ़बोर्ड के अनुकूल है।
-------------------------

## 17. Visualising training metrics

In [29]:
# Plot training metrics with error handling
def plot_training_metrics(metrics):
    """Plot training and evaluation metrics"""
    try:
        # Create epochs list
        epochs = list(range(1, len(metrics["train_loss"]) + 1))
        
        # Plot Loss Curves
        plt.figure(figsize=(12, 5))
        plt.plot(epochs, metrics["train_loss"], label="Training Loss", marker="o")
        plt.plot(epochs, metrics["eval_loss"], label="Validation Loss", marker="o")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.title("Training and Validation Loss")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig("training_loss.png")
        plt.show()
        
        # Plot Evaluation Metrics
        plt.figure(figsize=(12, 5))
        plt.plot(epochs, metrics["bleu"], label="BLEU", marker="o")
        plt.plot(epochs, metrics["rouge1"], label="ROUGE-1", marker="o")
        plt.plot(epochs, metrics["rouge2"], label="ROUGE-2", marker="o")
        plt.plot(epochs, metrics["rougeL"], label="ROUGE-L", marker="o")
        plt.plot(epochs, metrics["meteor"], label="METEOR", marker="o")
        plt.xlabel("Epochs")
        plt.ylabel("Score")
        plt.title("Evaluation Metrics")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig("evaluation_metrics.png")
        plt.show()
        
    except Exception as e:
        print(f"Error plotting metrics: {e}")
        # Try to plot whatever metrics are available
        for key, values in metrics.items():
            if values:
                plt.figure(figsize=(8, 4))
                plt.plot(list(range(1, len(values) + 1)), values, marker='o')
                plt.title(f"{key} over epochs")
                plt.xlabel("Epochs")
                plt.ylabel(key)
                plt.grid(True)
                plt.show()

# Plot metrics
plot_training_metrics(training_metrics)

NameError: name 'training_metrics' is not defined

In [30]:
def plot_training_metrics_extended(metrics=None, checkpoint_path=None):
    """
    Plot training metrics with error handling for missing keys
    """
    # Load metrics from checkpoint if not provided
    if metrics is None and checkpoint_path:
        try:
            print(f"Loading metrics from {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path)
            metrics = checkpoint.get('metrics', {})
            print(f"Available metrics keys: {list(metrics.keys())}")
        except Exception as e:
            print(f"Error loading metrics from checkpoint: {e}")
            print("Using empty metrics dictionary instead")
            metrics = {}
    
    # Default empty metrics if still None
    if metrics is None:
        metrics = {}
    
    # Check for required keys - use empty lists for missing keys
    required_keys = ['train_loss', 'eval_loss', 'bleu', 'rougeL', 'meteor']
    for key in required_keys:
        if key not in metrics:
            print(f"Warning: '{key}' not found in metrics, using empty list")
            metrics[key] = []
    
    # Create plots only if we have at least training loss
    if len(metrics.get("train_loss", [])) > 0:
        epochs = list(range(1, len(metrics["train_loss"])+1))
        
        # Create summary data with safeguards
        summary_data = {
            'Epoch': epochs,
            'Train Loss': [f"{loss:.4f}" for loss in metrics['train_loss']],
        }
        
        # Add other metrics only if they exist with matching length
        for metric_name, display_name, format_str in [
            ('eval_loss', 'Eval Loss', "{:.4f}"),
            ('bleu', 'BLEU', "{:.2f}"),
            ('rougeL', 'ROUGE-L', "{:.2f}"),
            ('meteor', 'METEOR', "{:.2f}")
        ]:
            if len(metrics.get(metric_name, [])) == len(epochs):
                summary_data[display_name] = [format_str.format(val) for val in metrics[metric_name]]
            else:
                summary_data[display_name] = ["N/A"] * len(epochs)
        
        # Create and display a nice summary table
        df = pd.DataFrame(summary_data)
        display(df)
        
        # Plot losses
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(epochs, metrics['train_loss'], 'b-', label='Training Loss')
        
        if len(metrics.get('eval_loss', [])) == len(epochs):
            plt.plot(epochs, metrics['eval_loss'], 'r-', label='Evaluation Loss')
        
        plt.title('Training and Evaluation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        
        # Plot metrics
        plt.subplot(1, 2, 2)
        for metric_name, color, label in [
            ('bleu', 'g-', 'BLEU Score'),
            ('rougeL', 'm-', 'ROUGE-L Score'),
            ('meteor', 'c-', 'METEOR Score')
        ]:
            if len(metrics.get(metric_name, [])) == len(epochs):
                plt.plot(epochs, metrics[metric_name], color, label=label)
        
        plt.title('Translation Quality Metrics')
        plt.xlabel('Epoch')
        plt.ylabel('Score')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    else:
        print("No training metrics available to plot")

def visualize_translations(model, tokenizer, test_examples=None, num_examples=5):
    """
    Show actual translations from the model to qualitatively assess performance.
    """
    if test_examples is None:
        # If no examples provided, load a few from test set
        try:
            test_data = load_dataset("cfilt/iitb-english-hindi", split="test")
            test_examples = test_data.select(range(num_examples))
        except Exception as e:
            print(f"Failed to load test data: {e}")
            return
    
    model.eval()
    
    results = []
    for idx, example in enumerate(test_examples):
        source_text = example['translation']['en']
        reference = example['translation']['hi']
        
        # Tokenize and generate translation
        inputs = tokenizer(source_text, return_tensors="pt", padding=True, truncation=True).to(device)
        
        try:
            with torch.no_grad():
                outputs = model.generate(**inputs, max_length=128, num_beams=4)
                translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            results.append({
                'id': idx,
                'source': source_text,
                'reference': reference,
                'translation': translation
            })
        except Exception as e:
            print(f"Error translating example {idx}: {e}")
    
    # Display results in a more visually appealing way
    for idx, item in enumerate(results):
        print(f"\n--- Example {idx+1} ---")
        print(f"🇬🇧 English:   {item['source']}")
        print(f"🇮🇳 Reference: {item['reference']}")
        print(f"🤖 Model:     {item['translation']}")
        print("-" * 80)

In [31]:
# Load from latest checkpoint for partial results mid-training
plot_training_metrics_extended(checkpoint_path="./results/latest_checkpoint/training_state.pt")
visualize_translations(model, tokenizer, num_examples=50)

Loading metrics from ./results/latest_checkpoint/training_state.pt
Error loading metrics from checkpoint: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the do

In [32]:
# Run this after training completes to show the full training history
plot_training_metrics_extended(training_metrics)
visualize_translations(model, tokenizer, test_examples=test_data.select(range(10)))

NameError: name 'training_metrics' is not defined

In [ ]:
# At the end of your notebook
try:
    # Try to use training_metrics if available (from completed training)
    if 'training_metrics' in globals() and training_metrics:
        plot_training_metrics_extended(training_metrics)
    else:
        # Otherwise load from latest checkpoint
        plot_training_metrics_extended(checkpoint_path="./results/latest_checkpoint/training_state.pt")
except Exception as e:
    print(f"Error visualizing metrics: {e}")

In [ ]:
# At the end of your notebook
try:
    # First priority: use training_metrics if available (from completed training)
    if 'training_metrics' in globals() and training_metrics:
        print("Using metrics from completed training")
        plot_training_metrics_extended(training_metrics)
    else:
        # Second priority: check latest checkpoint
        checkpoint_path = "./results/latest_checkpoint/training_state.pt"
        if os.path.exists(checkpoint_path):
            print(f"Using metrics from latest checkpoint: {checkpoint_path}")
            plot_training_metrics_extended(checkpoint_path=checkpoint_path)
        else:
            # Third priority: find most recent interim metrics
            print("Looking for interim metrics files...")
            interim_files = []
            for root, dirs, files in os.walk("./results"):
                for file in files:
                    if file == "interim_metrics.pt" and "interim_metrics_epoch" in root:
                        # Extract epoch and step info from path
                        parts = root.split("epoch")[1].split("_step")
                        epoch = int(parts[0])
                        step = int(parts[1])
                        interim_files.append((epoch, step, os.path.join(root, file)))
            
            if interim_files:
                # Sort by epoch and step (descending) to get most recent
                interim_files.sort(reverse=True)
                most_recent = interim_files[0][2]
                print(f"Using most recent interim metrics: {most_recent}")
                plot_training_metrics_extended(checkpoint_path=most_recent)
            else:
                print("No training metrics found")
except Exception as e:
    print(f"Error visualizing metrics: {e}")

Using metrics from latest checkpoint: ./results/interim_metrics_epoch3_step309388/interim_metrics.pt
Loading metrics from ./results/interim_metrics_epoch3_step309388/interim_metrics.pt
Error loading metrics from checkpoint: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serializ